### Salary prediction using Regression

In [27]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import tensorflow as tf
from tensorflow.keras.models import Sequential #type: ignore
from tensorflow.keras.layers import Dense #type: ignore
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard #type: ignore
import datetime

In [2]:
data = pd.read_csv("bank.csv")

In [3]:
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


### Preprocessing

In [4]:
# Drop irrelevant columns
data = data.drop(["RowNumber", "CustomerId", "Surname"], axis=1)

In [5]:
# Encode the categorical variables
label_encoder_gender = LabelEncoder()
data["Gender"] = label_encoder_gender.fit_transform(data["Gender"])

In [8]:
onehot_encoding_geography = OneHotEncoder(handle_unknown="ignore")
geography_encoded = onehot_encoding_geography.fit_transform(data[["Geography"]])

In [9]:
geo_encoded_df = pd.DataFrame(geography_encoded.toarray(), columns=onehot_encoding_geography.get_feature_names_out(["Geography"])) #type: ignore

In [10]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [11]:
# Combine all the columns with original data
data = pd.concat([data.drop("Geography", axis=1), geo_encoded_df], axis=1)

In [12]:
# Split the data into features and target
X = data.drop("EstimatedSalary", axis=1)
y = data["EstimatedSalary"]

In [17]:
# Save the encoders and scaler 
with open("label_encoder_gender.pkl", "wb") as file:
    pickle.dump(label_encoder_gender, file)
    
with open("onehot_encoding_geography.pkl", "wb") as file:
    pickle.dump(onehot_encoding_geography, file)

In [14]:
# Split the data in train and test dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
# Scale the feaures
scaler = StandardScaler()

In [16]:
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [18]:
with open("scaler.pkl", "wb") as file:
    pickle.dump(scaler, file)

### ANN Regression

In [22]:
# Build our ANN model
model = Sequential(
    [
        Dense(64, activation="relu", input_shape=(X_train.shape[1],)), #type: ignore
        Dense(32, activation="relu"),
        Dense(1),
    ]
)

In [24]:
# Compile the model
model.compile(optimizer="adam", loss="mean_absolute_error", metrics=["mae"])

In [26]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_3 (Dense)             (None, 64)                832       
                                                                 
 dense_4 (Dense)             (None, 32)                2080      
                                                                 
 dense_5 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [28]:
# Setup the Tensorboard
log_dir = "regression_logs/fit/" + datetime.datetime.now().strftime("%d%m%Y-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [29]:
# Setup Early stopping
early_stopping_callback = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

In [30]:
# Training the model
history = model.fit(
    X_train, y_train, 
    validation_data=(X_test, y_test), 
    epochs=100,
    callbacks=[tensorflow_callback, early_stopping_callback]
)

Epoch 1/100


250/250 [==============================] - 2s 5ms/step - loss: 100373.8047 - mae: 100373.8047 - val_loss: 98500.1250 - val_mae: 98500.1250
Epoch 2/100
250/250 [==============================] - 1s 3ms/step - loss: 99554.7891 - mae: 99554.7891 - val_loss: 96828.4531 - val_mae: 96828.4531
Epoch 3/100
250/250 [==============================] - 1s 3ms/step - loss: 96611.2422 - mae: 96611.2422 - val_loss: 92458.7422 - val_mae: 92458.7422
Epoch 4/100
250/250 [==============================] - 1s 3ms/step - loss: 90809.5859 - mae: 90809.5859 - val_loss: 85244.6094 - val_mae: 85244.6094
Epoch 5/100
250/250 [==============================] - 1s 3ms/step - loss: 82495.9609 - mae: 82495.9609 - val_loss: 76127.2422 - val_mae: 76127.2422
Epoch 6/100
250/250 [==============================] - 1s 3ms/step - loss: 73047.9297 - mae: 73047.9297 - val_loss: 67117.7109 - val_mae: 67117.7109
Epoch 7/100
250/250 [==============================] - 1s 3ms/step - loss: 64333.7031 - mae: 64333.703

In [31]:
%load_ext tensorboard

In [34]:
%tensorboard --logdir regression_logs/fit

Reusing TensorBoard on port 6007 (pid 19576), started 0:00:06 ago. (Use '!kill 19576' to kill it.)

In [35]:
# Evaluate model on the test data
test_loss, test_mae = model.evaluate(X_test, y_test)
print(f"Test MAE: {test_mae}")
print(f"Test loss: {test_loss}")

63/63 [==============================] - 0s 2ms/step - loss: 50156.3711 - mae: 50156.3711
Test MAE: 50156.37109375
Test loss: 50156.37109375


In [36]:
model.save("regression_model.h5")

c:\Users\Aadarsh\miniconda3\envs\venv\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
